# Experiment A: Random vs Word2Vec Embeddings

## Purpose
Show that Word2Vec is doing the heavy lifting, not the neural network architecture.

## Setup
**Fixed settings:**
- Architecture: 1 hidden layer (128 neurons), ReLU
- Learning rate: 1e-3
- Batch size: 64
- Epochs: 10
- min_df: 0.0005, max_df: 0.5

## Two conditions to test:
1. **Word2Vec (baseline)**: Normal setup - load Word2Vec embeddings
2. **Random embeddings**: Random 300D vectors (optionally normalized)

## Setup (Run this once)
Load dataset and prepare preprocessing

In [6]:
# Tested for Python=3.10
import torch
from sklearn.feature_extraction.text import CountVectorizer
from datasets import load_from_disk, load_dataset
import gensim
import os
import pandas as pd
import numpy as np

# Load dataset
if os.path.exists("./imdb_dataset"):
    imdb = load_from_disk("./imdb_dataset")
else:
    imdb = load_dataset("imdb")
    imdb.save_to_disk("./imdb_dataset")

reviews_train = imdb['train']["text"]
reviews_test = imdb['test']["text"]

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using {device} device")

seeds = [42, 123, 456, 789, 1011]
print(f"Seeds to use: {seeds}")

Using cpu device
Seeds to use: [42, 123, 456, 789, 1011]


In [7]:
# Preprocess data (fixed for all conditions)
dictionary = CountVectorizer(min_df=0.0005, max_df=0.5).fit(reviews_train)
vocab_size = len(dictionary.vocabulary_)
print(f"Vocabulary size: {vocab_size}")

reviews_train_dict = dictionary.transform(reviews_train)
reviews_test_dict = dictionary.transform(reviews_test)

reviews_train_dt = torch.from_numpy(reviews_train_dict.todense())
reviews_test_dt = torch.from_numpy(reviews_test_dict.todense())

print(f"Train shape: {reviews_train_dt.shape}")
print(f"Test shape: {reviews_test_dt.shape}")

Vocabulary size: 15862
Train shape: torch.Size([25000, 15862])
Test shape: torch.Size([25000, 15862])


In [8]:
# Training and testing functions
def train_loop(dataloader, model, loss_fn, optimizer):
    model.train()
    train_loss = 0
    for X, y in dataloader:
        X, y = X.to(device), y.to(device)
        pred = model(X)
        loss = loss_fn(pred, y)
        train_loss += loss.item()
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
    return train_loss / len(dataloader)

def test_loop(dataloader, model, loss_fn):
    model.eval()
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    test_loss /= len(dataloader)
    correct /= len(dataloader.dataset)
    return test_loss, correct

## Experiment A: Random vs Word2Vec Embeddings

Testing two embedding conditions with multiple seeds

In [9]:
# Hyperparameters (fixed)
batch_size = 64
epochs = 10
lr = 1e-3

# Prepare dataloaders
train_dataset = torch.utils.data.TensorDataset(reviews_train_dt, torch.tensor(imdb['train']['label'], dtype=torch.long))
test_dataset = torch.utils.data.TensorDataset(reviews_test_dt, torch.tensor(imdb['test']['label'], dtype=torch.long))
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

print(f"Train loader: {len(train_loader)} batches")
print(f"Test loader: {len(test_loader)} batches")

Train loader: 391 batches
Test loader: 391 batches


In [10]:
# Load Word2Vec embeddings (for condition 1)
print("Loading Word2Vec model...")
model_w2v = gensim.models.KeyedVectors.load_word2vec_format('../.././GoogleNews-vectors-negative300.bin.gz', binary=True)
print("Word2Vec model loaded!")

# Create Word2Vec embedding matrix
word_embs_w2v = torch.zeros((vocab_size, 300), dtype=torch.float)
words_not_found = 0
for k, i in dictionary.vocabulary_.items():
    try:
        word_embs_w2v[i, :] = torch.tensor(model_w2v[k], dtype=torch.float)
    except KeyError:
        words_not_found += 1

print(f"Words not found in Word2Vec: {words_not_found} out of {vocab_size}")
print(f"Word2Vec embedding matrix shape: {word_embs_w2v.shape}")

Loading Word2Vec model...
Word2Vec model loaded!
Words not found in Word2Vec: 1145 out of 15862
Word2Vec embedding matrix shape: torch.Size([15862, 300])


In [11]:
# Create random embedding matrix
torch.manual_seed(999)  # Fixed seed for reproducibility of random embeddings
word_embs_random = torch.randn((vocab_size, 300), dtype=torch.float)
word_embs_random = word_embs_random / word_embs_random.norm(dim=1, keepdim=True)

print(f"Random embedding matrix shape: {word_embs_random.shape}")
print(f"Random embeddings mean norm: {word_embs_random.norm(dim=1).mean():.4f}")
print(f"Word2Vec embeddings mean norm: {word_embs_w2v.norm(dim=1).mean():.4f}")

Random embedding matrix shape: torch.Size([15862, 300])
Random embeddings mean norm: 1.0000
Word2Vec embeddings mean norm: 2.7618


In [12]:
# Function to create embedded reviews
def create_embeddings(review_counts, embedding_matrix):
    """Create embedded reviews from count vectors and embedding matrix"""
    review_counts_normalized = review_counts / review_counts.sum(axis=1, keepdim=True)
    review_counts_normalized[torch.isnan(review_counts_normalized)] = 0
    review_embs = review_counts_normalized @ embedding_matrix
    return review_embs

# Verify
test_embs = create_embeddings(reviews_train_dt.float(), word_embs_w2v)
print(f"Sample embedded review shape: {test_embs[0].shape}")

Sample embedded review shape: torch.Size([300])


In [13]:
# Run experiment with both embedding types
embedding_types = {
    'Word2Vec': word_embs_w2v,
    'Random': word_embs_random
}

all_results = []

for emb_type, emb_matrix in embedding_types.items():
    print(f"\n{'='*70}")
    print(f"Embedding Type: {emb_type}")
    print(f"{'='*70}")
    
    # Create embedded datasets
    review_train_embs = create_embeddings(reviews_train_dt.float(), emb_matrix).to(device)
    review_test_embs = create_embeddings(reviews_test_dt.float(), emb_matrix).to(device)
    
    # Create new dataloaders with embeddings
    train_dataset_emb = torch.utils.data.TensorDataset(
        review_train_embs, 
        torch.tensor(imdb['train']['label'], dtype=torch.long).to(device)
    )
    test_dataset_emb = torch.utils.data.TensorDataset(
        review_test_embs, 
        torch.tensor(imdb['test']['label'], dtype=torch.long).to(device)
    )
    
    train_loader_emb = torch.utils.data.DataLoader(train_dataset_emb, batch_size=batch_size, shuffle=True, num_workers=0)
    test_loader_emb = torch.utils.data.DataLoader(test_dataset_emb, batch_size=batch_size, shuffle=False, num_workers=0)
    
    emb_accuracies = []
    epoch_accuracies = {seed: [] for seed in seeds}
    
    for seed in seeds:
        print(f"\n  Seed {seed}:")
        torch.manual_seed(seed)
        
        model = torch.nn.Sequential(
            torch.nn.Linear(300, 128),
            torch.nn.ReLU(),
            torch.nn.Linear(128, 2),
        ).to(device)
        
        loss_fn = torch.nn.CrossEntropyLoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        
        best_accuracy = 0
        
        for epoch in range(epochs):
            train_loss = train_loop(train_loader_emb, model, loss_fn, optimizer)
            test_loss, accuracy = test_loop(test_loader_emb, model, loss_fn)
            
            epoch_accuracies[seed].append(accuracy)
            all_results.append({
                'embedding_type': emb_type,
                'seed': seed,
                'epoch': epoch + 1,
                'accuracy': accuracy
            })
            
            if accuracy > best_accuracy:
                best_accuracy = accuracy
        
        emb_accuracies.append(best_accuracy)
        print(f"    Best Accuracy: {best_accuracy:.4f}")
    
    mean_acc = np.mean(emb_accuracies)
    std_acc = np.std(emb_accuracies)
    
    print(f"\n  {emb_type} Mean Accuracy: {mean_acc:.4f} ± {std_acc:.4f}")


Embedding Type: Word2Vec

  Seed 42:
    Best Accuracy: 0.8606

  Seed 123:
    Best Accuracy: 0.8604

  Seed 456:
    Best Accuracy: 0.8582

  Seed 789:
    Best Accuracy: 0.8605

  Seed 1011:
    Best Accuracy: 0.8594

  Word2Vec Mean Accuracy: 0.8598 ± 0.0009

Embedding Type: Random

  Seed 42:
    Best Accuracy: 0.7432

  Seed 123:
    Best Accuracy: 0.7436

  Seed 456:
    Best Accuracy: 0.7434

  Seed 789:
    Best Accuracy: 0.7434

  Seed 1011:
    Best Accuracy: 0.7429

  Random Mean Accuracy: 0.7433 ± 0.0002


In [14]:
# Create results dataframe
results_df = pd.DataFrame(all_results)

# Summary by embedding type
summary = results_df.groupby('embedding_type')['accuracy'].agg(['mean', 'std', 'min', 'max'])
print("\n" + "="*70)
print("SUMMARY: Random vs Word2Vec Embeddings")
print("="*70)
print(summary)

# Final epoch accuracies only
final_epoch_results = results_df[results_df['epoch'] == epochs].copy()
final_summary = final_epoch_results.groupby('embedding_type')['accuracy'].agg(['mean', 'std'])
print("\n" + "="*70)
print(f"FINAL ACCURACY (Epoch {epochs})")
print("="*70)
print(final_summary)

# Display detailed results
print("\n" + "="*70)
print("DETAILED RESULTS: All epochs for all seeds")
print("="*70)
print(results_df.to_string(index=False))


SUMMARY: Random vs Word2Vec Embeddings
                    mean       std      min      max
embedding_type                                      
Random          0.740138  0.004779  0.72436  0.74356
Word2Vec        0.853424  0.006096  0.83552  0.86056

FINAL ACCURACY (Epoch 10)
                    mean       std
embedding_type                    
Random          0.742544  0.000620
Word2Vec        0.858232  0.002699

DETAILED RESULTS: All epochs for all seeds
embedding_type  seed  epoch  accuracy
      Word2Vec    42      1   0.84068
      Word2Vec    42      2   0.85004
      Word2Vec    42      3   0.85452
      Word2Vec    42      4   0.84876
      Word2Vec    42      5   0.85668
      Word2Vec    42      6   0.85692
      Word2Vec    42      7   0.85960
      Word2Vec    42      8   0.85828
      Word2Vec    42      9   0.84980
      Word2Vec    42     10   0.86056
      Word2Vec   123      1   0.83960
      Word2Vec   123      2   0.85016
      Word2Vec   123      3   0.85404
     

In [15]:
# Save results to CSV
results_df.to_csv('results_embeddings.csv', index=False)
print("Results saved to results_embeddings.csv")

# Save summary
final_summary.to_csv('results_embeddings_summary.csv')
print("Summary saved to results_embeddings_summary.csv")

Results saved to results_embeddings.csv
Summary saved to results_embeddings_summary.csv
